## **Activity 1. NumPy**
---

### Establishing Baseline Data

In [3]:
import numpy as np
import pandas as pd

df = pd.read_csv("../datasets/kickstarter_projects.csv")
df.columns = df.columns.str.strip()

df_transformed = df.copy()
df_transformed[["Launched", "Deadline"]] = df_transformed[["Launched", "Deadline"]].apply(pd.to_datetime)
numeric_values = df_transformed[["Goal", "Pledged", "Backers"]].to_numpy()

missing = df[["Goal", "Pledged", "Backers"]].isna().sum()
print(missing)
print("Total rows removed for missing values:", missing.sum())

df_transformed[["Launched", "Deadline", "Goal", "Pledged", "Backers", "State"]].head(10)

Goal       0
Pledged    0
Backers    0
dtype: int64
Total rows removed for missing values: 0


,Launched,Deadline,Goal,Pledged,Backers,State
0,2009-04-21 21:02:48,2009-05-31,1000,625,30,Failed
1,2009-04-23 00:07:53,2009-07-20,80000,22,3,Failed
2,2009-04-24 21:52:03,2009-05-03,20,35,3,Successful
3,2009-04-25 17:36:21,2009-07-14,99,145,25,Successful
4,2009-04-27 14:10:39,2009-05-26,1900,387,10,Failed
5,2009-04-28 13:55:41,2009-05-16,3000,3329,110,Successful
6,2009-04-29 02:04:21,2009-05-29,200,41,3,Failed
7,2009-04-29 02:58:50,2009-05-29,500,563,18,Successful
8,2009-04-29 04:37:37,2009-06-01,500,0,0,Canceled
9,2009-04-29 05:26:32,2009-05-16,300,15,2,Failed


**Explanation:**  
This dataset is a Kickstarter project data containing 374,853 total projects that has numerical columns (Goal, Pledged, and Backers) stored as int64 and also contains no missing values. The numeric columns were converted to a NumPy array for analysis.

### Task 1. Inspect the Numerical Array
---

In [4]:
numeric_labels = ["Goal", "Pledged", "Backers"]

def report_numeric(numeric_val):
    return [
        len(numeric_val),
        numeric_val.shape,
        numeric_val.size,
        numeric_val.dtype,
        np.min(numeric_val),
        np.max(numeric_val),
        np.mean(numeric_val),
        np.median(numeric_val),
        np.std(numeric_val),
    ]

combined = pd.DataFrame(
    {label: report_numeric(col) for col, label in zip(numeric_values.T, numeric_labels)},
    index=[
        "Number of Observations", "Shape", "Size", "Data Type",
        "Minimum", "Maximum", "Mean", "Median", "Standard Deviation",
    ],
)

combined.style.format(precision=2)

,Goal,Pledged,Backers
Number of Observations,374853,374853,374853
Shape,"(374853,)","(374853,)","(374853,)"
Size,374853,374853,374853
Data Type,int64,int64,int64
Minimum,0,0,0
Maximum,166361391,20338986,219382
Mean,45863.78,9121.07,106.69
Median,5500.00,625.00,12.00
Standard Deviation,1158776.63,91320.42,911.72


In [5]:
zero_goal = df_transformed[df_transformed["Goal"] == 0]

print("Projects with Goal == 0:", len(zero_goal))
print("  ...of which Pledged > 0 (would give infinite):", (zero_goal["Pledged"] > 0).sum())
print("  ...of which Pledged == 0 (would give NaN):", (zero_goal["Pledged"] == 0).sum())

Projects with Goal == 0: 4
  ...of which Pledged > 0 (would give infinite): 2
  ...of which Pledged == 0 (would give NaN): 2


**Explanation:** 
The three numeric columns each hold 374,853 observations where in all three, the mean is far larger than the median and the standard deviation is very large (for example, Goal has a median of 5,500 but a mean of about 45,864 and a standard deviation of about 1.16 million). This signifies that the data is strongly right-skewed: most projects are small, while a few extreme projects pull the averages up (the largest Goal is about 166 million).  

The minimum Goal of 0 is also suspicious since a project cannot meaningfully have a zero funding target. Only 4 projects have Goal = 0, but 2 of them have Pledged > 0, which makes *Pledged / Goal* infinite (the other 2 give 0 / 0 = NaN). These rows are handled in Task 2 before computing *percent_to_goal*.

### Task 2. Vectorized Computation
---

#### Domain-Specific Calculation

In [6]:
# Campaign length in days (Deadline - Launched). NOTE: this is the planned duration of the campaign, NOT "days before the deadline" - the dataset has no date for when a goal was reached or a project was stopped.
df_transformed["campaign_days"] = (df_transformed["Deadline"] - df_transformed["Launched"]).dt.days

# Live counts are excluded since they are considered still ongoing
df_transformed_not_success = df_transformed[
    df_transformed["State"].isin(["Failed", "Canceled", "Suspended"])
].copy()

# i identified zero-goal projects that make Pledged / Goal undefined (inf or NaN) 
zero_goal_ns = df_transformed_not_success[df_transformed_not_success["Goal"] == 0]
df_transformed_not_success = df_transformed_not_success[df_transformed_not_success["Goal"] > 0].copy()

df_transformed_not_success_np = df_transformed_not_success[["Goal", "Pledged"]].to_numpy()

# this calculates how close each campaign got to its funding target in percentage (This uses numpy method divide)
df_transformed_not_success["percent_to_goal"] = np.divide(df_transformed_not_success_np[:, 1], df_transformed_not_success_np[:, 0]) * 100

# this is a boolean that marks every project with a percentage for 100% or higher as True, otherwise False
reached = df_transformed_not_success["percent_to_goal"] >= 100

# i also checked the not successful, yet pledged >= goal. then i split by State to see which cases are explainable
failed_but_achieved_goal_df = df_transformed_not_success[reached & (df_transformed_not_success["State"] == "Failed")]
stopped_but_achieved_goal_df = df_transformed_not_success[reached & df_transformed_not_success["State"].isin(["Canceled", "Suspended"])]
truly_unsuccessful = df_transformed_not_success[~reached]

In [7]:
summary = pd.DataFrame({
    "Dataset": [
        "Full transformed DataFrame",
        "Not-Successful (Failed/Canceled/Suspended, Goal > 0)",
        "Pledged >= 100% of goal but State = Failed (Anomaly)",
        "Pledged >= 100% of goal but Canceled/Suspended",
        "Truly Unsuccessful (< 100% of goal)",
    ],
    "Rows": [
        df_transformed.shape[0],
        df_transformed_not_success.shape[0],
        failed_but_achieved_goal_df.shape[0],
        stopped_but_achieved_goal_df.shape[0],
        truly_unsuccessful.shape[0],
    ],
    "Columns": [
        df_transformed.shape[1],
        df_transformed_not_success.shape[1],
        failed_but_achieved_goal_df.shape[1],
        stopped_but_achieved_goal_df.shape[1],
        truly_unsuccessful.shape[1],
    ],
    "Shape": [
        df_transformed.shape,
        df_transformed_not_success.shape,
        failed_but_achieved_goal_df.shape,
        stopped_but_achieved_goal_df.shape,
        truly_unsuccessful.shape,
    ],
})

print("Excluded -> Live projects:", (df_transformed["State"] == "Live").sum(),
      "| zero-goal not-successful projects:", len(zero_goal_ns))

summary

Excluded -> Live projects: 2798 | zero-goal not-successful projects: 2


,Dataset,Rows,Columns,Shape
0,Full transformed DataFrame,374853,12,"(374853, 12)"
1,"Not-Successful (Failed/Canceled/Suspended, Goa...",238202,13,"(238202, 13)"
2,Pledged >= 100% of goal but State = Failed (An...,9,13,"(9, 13)"
3,Pledged >= 100% of goal but Canceled/Suspended,989,13,"(989, 13)"
4,Truly Unsuccessful (< 100% of goal),237204,13,"(237204, 13)"


**Explanation:**  
For this, I wanted to determine how many unsuccessful projects were close to isolate non-successful campaigns and also know if how close each project actually came to its target, while eliminating dataset noise and isolating explainable edge cases.  

I first calculated the planned campaign duration in days to establish a standard timeframe, then filtered out ongoing "Live" projects and cleaned out invalid zero-goal entries to prevent mathematical errors during percentage calculations. By computing the percentage of target funding achieved for every non-successful campaign, I was able to segment the data into three distinct subsets: 
- projects that met or exceeded 100% of their goal but were marked as "Failed" (data anomalies),
- projects that reached 100% but were terminated early ("Canceled" or "Suspended"),
- and projects that were truly unsuccessful (<100%)

#### **Anomaly Detected Calculation**  
This part summarizes the checks that I had to do wherein i had to determine projects that were tagged as a not successful but reached their goal which are considered anomalous rows.

In [8]:
# this asks which states account for the "not successful but reached the goal" rows
reached_all = df_transformed_not_success[reached]
reached_all["State"].value_counts().rename("Projects").to_frame()

,Projects
State,
Canceled,697
Suspended,292
Failed,9


In [9]:
failed_but_achieved_goal_df[["Name", "Goal", "Pledged", "Backers", "State", "percent_to_goal"]].assign(
    percent_to_goal=lambda d: d["percent_to_goal"].round(2)
)

,Name,Goal,Pledged,Backers,State,percent_to_goal
2295,The new face of Canadian Science Fiction,10000,10797,66,Failed,107.97
5754,"POPE! an epic musical, opening at NY Fringe Fe...",3600,3600,30,Failed,100.00
25983,From Neurons to Nirvana: The Great Medicines,40000,40000,113,Failed,100.00
33823,Music Empowers - A Day with the Accende Ensemble,3000,3088,20,Failed,102.93
62467,Tye Gazoo,3000,3000,24,Failed,100.00
86441,"""Balance""",23000,23020,141,Failed,100.09
108392,Key Wrench - Unlock Your Life!,6000,6183,208,Failed,103.05
151769,Egg Salad,10,10,1,Failed,100.00
259754,The 5th Annual 90 Square Arts Festival,12000,12000,77,Failed,100.00


In [10]:
stopped_but_achieved_goal_df[["Name", "Goal", "Pledged", "Backers", "State", "campaign_days", "percent_to_goal"]].head().assign(
    percent_to_goal=lambda d: d["percent_to_goal"].round(2)
)

,Name,Goal,Pledged,Backers,State,campaign_days,percent_to_goal
58,Save the Ramzi (Canceled),1000,1190,18,Canceled,31,119.00
1106,Help Sex Piano Fund an EP (Canceled),750,1110,32,Canceled,76,148.00
3255,Tiny Urban Park (Canceled),650,1260,39,Canceled,13,193.85
6152,Pieces of My Soul: Scattered Around Chicago. (...,60,160,4,Canceled,89,266.67
7604,Microfiche releasing Jetpacks and Laser Beams ...,1500,1905,24,Canceled,35,127.00


**Explanation:**  

Of the 238,202 not-successful projects, 998 reached 100% or more of their goal. The breakdown by State explains almost all of them: 697 were **Canceled** and 292 **Suspended**, which is possible because a creator can cancel a campaign, or Kickstarter can suspend it, after the goal was met. Only **9 Failed** projects reached or exceeded their goal, and this is the genuine anomaly. Five of them pledged exactly 100% (for example 3,600 of 3,600), and the others only slightly exceeded it. I could not determine the reason why these are the results but most likely the are pledges that were dropped or failed to collect after the snapshot. The remaining 237,204 projects are truly unsuccessful (below 100% of goal).

### Task 3. Boolean Masking
---  

In [11]:
# this asks which unsuccessful projects came closest to their goal
np_not_success = truly_unsuccessful[["Goal", "Pledged", "percent_to_goal", "Backers"]].to_numpy().T
print("np_not_success shape:", np_not_success.shape, "| NaN present:", np.isnan(np_not_success).any())

# Condition: percent_to_goal must be above the 75th percentile 
percent_to_goal = np_not_success[2, :]                 # row 2 = percent_to_goal
p75  = np.percentile(percent_to_goal, 75)
mask = percent_to_goal > p75

# the boolean mask selects the top ~25% 
above_75th_percentile = np_not_success[:, mask]

# check
n_total  = np_not_success.shape[1]
n_top25  = above_75th_percentile.shape[1]
pct_kept = n_top25 / n_total * 100

summary = pd.DataFrame({
    "Metric": [
        "Total unsuccessful records",
        "75th percentile of percent_to_goal",
        "Records above 75th percentile",
        "Percent of records kept",
        "Mean percent_to_goal (top 25%)",
        "Std dev percent_to_goal (top 25%)",
        "Min percent_to_goal (top 25%)",
        "Max percent_to_goal (top 25%)",
    ],
    "Value": [
        n_total,
        round(p75, 2),
        n_top25,
        f"{pct_kept:.2f}%",
        round(np.mean(above_75th_percentile[2, :]), 2),
        round(np.std(above_75th_percentile[2, :]), 2),
        round(np.min(above_75th_percentile[2, :]), 2),
        round(np.max(above_75th_percentile[2, :]), 2),
    ],
})
summary

np_not_success shape: (4, 237204) | NaN present: False


,Metric,Value
0,Total unsuccessful records,237204
1,75th percentile of percent_to_goal,10.8
2,Records above 75th percentile,59284
3,Percent of records kept,24.99%
4,Mean percent_to_goal (top 25%),30.01
5,Std dev percent_to_goal (top 25%),17.65
6,Min percent_to_goal (top 25%),10.8
7,Max percent_to_goal (top 25%),99.96


**Explanation:**  
The threshold is the 75th percentile of *percent_to_goal* among the truly unsuccessful projects, which results in 10.80%. I prefer a percentile over a fixed number because the data is heavily skewed, and a percentile adapts to the data.  

The mask isolates the top quarter of unsuccessful projects, which are the ones that came closest to their goals. It selected 59,284 out of 237,204 records, that is about 25% (slightly under 25.00% because the rule is a strict "above" the threshold). These projects averaged only about 30% of their goal (standard deviation about 17.65), and none reached 100% (the maximum is 99.96%), which is expected since projects at or above 100% were set aside in Example 2.

### Task 4. Vectorization vs. Iteration
---

In [12]:
import time

row2 = np.tile(np_not_success[2, :], 20)
p75  = np.percentile(row2, 75)
mask = row2 > p75                      # same strict ">" rule as Task 3
data = row2.tolist()                   # plain Python list for the loop version
print(f"Elements timed: {len(row2):,}")

n_runs = 5
loop_times, numpy_times = [], []

for _ in range(n_runs):
    # Method 1: Python list comprehension
    start = time.perf_counter()
    top_list = [i for i in data if i > p75]
    end = time.perf_counter()
    loop_times.append(end - start)

    # Method 2: NumPy boolean masking
    start = time.perf_counter()
    top_np = row2[mask]
    end = time.perf_counter()
    numpy_times.append(end - start)

loop_time  = np.mean(loop_times)
numpy_time = np.mean(numpy_times)

# Verify equivalence 
equivalent = np.allclose(np.asarray(top_list), top_np)
print(f"Results equivalent: {equivalent}")
print(f"Loop length:  {len(top_list):,}")
print(f"NumPy length: {len(top_np):,}")

speedup = loop_time / numpy_time

results = pd.DataFrame({
    "Metric": [
        "Loop time, mean of 5 runs (s)",
        "NumPy time, mean of 5 runs (s)",
        "Speedup (loop / numpy)",
        "Results equivalent (np.allclose)",
    ],
    "Value": [
        f"{loop_time:.6f}",
        f"{numpy_time:.6f}",
        f"{speedup:.2f}x",
        str(equivalent),
    ],
})
results

Elements timed: 4,744,080
Results equivalent: True
Loop length:  1,185,680
NumPy length: 1,185,680


,Metric,Value
0,"Loop time, mean of 5 runs (s)",0.095721
1,"NumPy time, mean of 5 runs (s)",0.011340
2,Speedup (loop / numpy),8.44x
3,Results equivalent (np.allclose),True


**Explanation:**  
I filtered values above the 75th percentile with a Python list comprehension and with a NumPy boolean mask on a larger array (the Task 3 row tiled 20 times, about 4.7 million values), and averaged five timed runs. NumPy was many times faster (the table above shows the speedup for this run; exact timings vary by machine) because it ran the loop in compiled C code over same-type numbers stored contiguously in memory, while a Python loop pays interpreter overhead on every item. *np.allclose* confirmed that both methods returned identical values and the same number of elements.

### Task 5. Interpretation
---

**1. What did the NumPy analysis reveal about your selected numerical variable?**  
The numbers are heavily right-skewed. Most projects ask for little money, raise little, and attract few backers, while a few extreme projects distort the means. Among the truly unsuccessful projects, even the top quarter by *percent_to_goal* averaged only about 30% of their goal. The analysis also exposed data-quality issues: zero-goal projects, and a small group of Failed projects (9) that appear to have reached their goal.

**2. What is the role of vectorization in this analysis?**  
Vectorization allows percentages to be computed, boolean masks to be built, and millions of records summarized in single expressions without loops, making the code shorter, more readable, and faster.

**3. Why might vectorization become more important as the dataset grows larger?**  
Since Python handles each item one at a time, loop time also grows with every row whereas NumPy processes whole arrays in compiled code over contiguous same-type memory. In data science and machine learning, proccessing millions of rows using loops becomes impractical especially when the data grows because its gap widens even more.

**4. What are the limitations of a plain NumPy array compared with a labeled Pandas Series or DataFrame?**  
For NumPy arrays, specific tabular positions or indexing is quite complicated since it has no column names or row labels because it only holds a single data type. It also has limited built-in support for missing values, aggregating, joining, and cleaning. Pandas, however, adds labels and mixed column types on top of NumPy.